# 11 · Multistate Markov model — NP1PAIN transitions by DBS

Model pain as a 4-state process: {0=None, 1=Mild, 2=Moderate, 3≥Severe}. Use `msm::msm()` to estimate transition intensities (per month) and compare DBS vs Never-DBS. Output: transition matrices, sojourn times, and hazard ratios for DBS on each transition.

In [1]:
source("helpers/pain_helpers.R")
suppressPackageStartupMessages({ library(dplyr); library(tidyr); library(ggplot2); library(msm) })
df <- readRDS(file.path(OUT_OBJ, "pain_long.rds"))

# Collapse NP1PAIN 3 and 4 into one 'Severe' state (state 3)
msm_df <- df %>% dplyr::filter(!is.na(NP1PAIN), time_pos >= 0) %>%
  dplyr::mutate(state = pmin(NP1PAIN, 3) + 1L) %>%  # 1..4
  dplyr::arrange(PATNO, time_pos_months) %>%
  dplyr::group_by(PATNO, time_pos_months) %>%
  dplyr::slice_head(n = 1) %>% dplyr::ungroup() %>%
  dplyr::select(PATNO, time_pos_months, state, will_receive_dbs)

cat("Rows:", nrow(msm_df), "  patients:", dplyr::n_distinct(msm_df$PATNO), "\n")
print(dplyr::count(msm_df, state, name = "n"))
# Keep only patients with ≥ 2 visits
keep <- msm_df %>% dplyr::count(PATNO) %>% dplyr::filter(n >= 2) %>% dplyr::pull(PATNO)
msm_df <- msm_df %>% dplyr::filter(PATNO %in% keep)
cat("After ≥2-visit filter:", dplyr::n_distinct(msm_df$PATNO), "patients\n")

Rows: 1219   patients: 170 


# A tibble: 4 × 2
  state     n
  <dbl> <int>
1     1   441
2     2   467
3     3   169
4     4   142


After ≥2-visit filter: 159 patients


In [2]:
# Allowed transitions: one-step up or down, plus self.  (Disallow 1↔3 direct jumps — msm will allow them if we let it, but restricting accelerates fit.)
Q <- rbind(
  c(0, 0.05, 0,    0   ),
  c(0.05, 0, 0.05, 0   ),
  c(0,    0.05, 0, 0.05),
  c(0,    0,    0.05, 0)
)
diag(Q) <- -rowSums(Q)
Qcrude <- msm::crudeinits.msm(state ~ time_pos_months, PATNO, data = msm_df,
                              qmatrix = Q)
print(Qcrude)

            [,1]        [,2]        [,3]        [,4]
[1,] -0.04767888  0.04767888  0.00000000  0.00000000
[2,]  0.03972268 -0.06289424  0.02317156  0.00000000
[3,]  0.00000000  0.04347800 -0.07536187  0.03188387
[4,]  0.00000000  0.00000000  0.03074717 -0.03074717


In [3]:
# Fit msm with DBS as covariate on all intensities
fit_msm <- tryCatch(
  msm::msm(state ~ time_pos_months, subject = PATNO,
           data = msm_df, qmatrix = Qcrude,
           covariates = ~ will_receive_dbs,
           control = list(fnscale = 4000, maxit = 500)),
  error = function(e) { cat("msm fit failed:", conditionMessage(e), "\n"); NULL })

if (!is.null(fit_msm)) {
  print(fit_msm)
  # Hazard ratios for DBS on each transition
  hr <- msm::hazard.msm(fit_msm)
  print(hr)
  # Transition probabilities at 12 months by arm
  p_ctl <- msm::pmatrix.msm(fit_msm, t = 12, covariates = list(will_receive_dbs = FALSE))
  p_dbs <- msm::pmatrix.msm(fit_msm, t = 12, covariates = list(will_receive_dbs = TRUE))
  cat("\n12-month transition probabilities — Never-DBS:\n"); print(round(p_ctl, 3))
  cat("\n12-month transition probabilities — DBS:\n"); print(round(p_dbs, 3))
  # Sojourn times by arm
  s_ctl <- msm::sojourn.msm(fit_msm, covariates = list(will_receive_dbs = FALSE))
  s_dbs <- msm::sojourn.msm(fit_msm, covariates = list(will_receive_dbs = TRUE))
  cat("\nSojourn times (months) — Never-DBS:\n"); print(s_ctl)
  cat("\nSojourn times (months) — DBS:\n"); print(s_dbs)
  save_object(fit_msm, "msm_fit")
  save_object(list(hr = hr, p_ctl = p_ctl, p_dbs = p_dbs,
                   s_ctl = s_ctl, s_dbs = s_dbs), "msm_outputs")
}


Call:
msm::msm(formula = state ~ time_pos_months, subject = PATNO,     data = msm_df, qmatrix = Qcrude, covariates = ~will_receive_dbs,     control = list(fnscale = 4000, maxit = 500))

Maximum likelihood estimates
Baselines are with covariates set to their means

Transition intensities with hazard ratios for each covariate
                  Baseline                  will_receive_dbsTRUE  
State 1 - State 1 -12.97 ( -17.109, -9.832)                       
State 1 - State 2  12.97 (   9.832, 17.109) 0.7786 (0.44664,1.357)
State 2 - State 1  11.85 (   8.927, 15.743) 0.8685 (0.49203,1.533)
State 2 - State 2 -29.69 ( -41.737,-21.117)                       
State 2 - State 3  17.83 (  10.310, 30.846) 1.9828 (0.66592,5.904)
State 3 - State 2  47.75 (  26.787, 85.132) 1.4372 (0.45411,4.549)
State 3 - State 3 -92.67 (-141.296,-60.781)                       
State 3 - State 4  44.92 (  18.872,106.913) 0.4494 (0.07809,2.586)
State 4 - State 3  49.10 (  20.577,117.158) 0.5276 (0.09120,3.052)
Sta

In [4]:
if (!is.null(fit_msm)) {
  # HR plot
  hr_tbl <- dplyr::bind_rows(lapply(names(hr), function(nm) {
    as.data.frame(hr[[nm]]) %>% tibble::rownames_to_column("transition") %>%
      dplyr::mutate(covariate = nm)
  })) %>% dplyr::rename(HR = `HR`, lci = `L`, uci = `U`) %>%
    dplyr::filter(stringr::str_detect(covariate, "will_receive_dbs"))
  print(hr_tbl)
  save_table(hr_tbl, "msm_hr_dbs")

  p_hr <- ggplot(hr_tbl, aes(x = HR, y = transition)) +
    geom_vline(xintercept = 1, linetype = "dashed", colour = "grey40") +
    geom_errorbar(aes(xmin = lci, xmax = uci), width = 0.15, orientation = "y") +
    geom_point(size = 3, colour = "#1b9e77") +
    scale_x_log10() +
    labs(title = "DBS hazard ratios on NP1PAIN state transitions",
         x = "HR (log scale)", y = NULL) +
    theme_classic(base_size = 12) +
    theme(plot.title = element_text(face = "bold", hjust = 0.5))
  p_hr
  save_fig(p_hr, "Fig23_msm_HR_DBS", width = 7, height = 4)
}

         transition        HR        lci      uci            covariate
1 State 1 - State 2 0.7785764 0.44663836 1.357208 will_receive_dbsTRUE
2 State 2 - State 1 0.8685333 0.49202742 1.533146 will_receive_dbsTRUE
3 State 2 - State 3 1.9828331 0.66591665 5.904083 will_receive_dbsTRUE
4 State 3 - State 2 1.4372189 0.45410906 4.548683 will_receive_dbsTRUE
5 State 3 - State 4 0.4494277 0.07809367 2.586448 will_receive_dbsTRUE
6 State 4 - State 3 0.5276029 0.09120302 3.052145 will_receive_dbsTRUE
